# Yapay Zeka Haber Denetçisi

DSPy, yapay zeka modellerini manuel ve kırılgan "prompt" metinleriyle yönetmek yerine, onları birer yazılım fonksiyonu gibi programlamayı sağlayan akademik bir framework'tür.

In [32]:
!pip install -q dspy-ai gradio litellm

In [34]:
import dspy
import gradio as gr
from google.colab import userdata
import datetime
from dspy.teleprompt import BootstrapFewShot

In [35]:
try:
    # Colab'tan'GROQ_API_KEY' çek
    groq_key = userdata.get('GROQ_API_KEY')

    # DSPy'ın LM arayüzü sayesinde Llama 3.1 modelini Groq motoruyla bağlar
    # Groq seçimi, saniyede yüzlerce kelime (token) üreterek yüksek performanslı çıkarım sağlar
    language_model = dspy.LM('groq/llama-3.1-8b-instant', api_key=groq_key)

    # dil modelini tüm DSPy programı için 'varsayılan beyin'olarak atar
    dspy.settings.configure(lm=language_model)
    print(f"[{datetime.datetime.now().strftime('%H:%M:%S')}] Sistem Hazır: Groq/Llama 3.1 Bağlandı!")
except Exception as e:
    print(f"❌ Bağlantı Hatası: {e}")

[20:50:12] Sistem Hazır: Groq/Llama 3.1 Bağlandı!


In [19]:
# try:
#     # Colabtan'OPENAI_API_KEY' çek
#     api_key = userdata.get('OPENAI_API_KEY')

#     # dspy.LM ile OpenAI modelini tanımlıyoruz
#     # OpenAI modelini 'openai/gpt-4o-mini' formatında çağırıyoruz
#     language_model = dspy.LM('openai/gpt-4o-mini', api_key=api_key)

#     # tanımlanan modeli DSPy'ın varsayılan dil motoru olarak ayarlar.
#     dspy.settings.configure(lm=language_model)

#     print(f"[{datetime.datetime.now().strftime('%H:%M:%S')}]OpenAI (GPT-4o-mini) Başarıyla Bağlandı!")
# except Exception as e:
#     print(f"BAĞLANTI HATASI")
#     print(f"Detay: {str(e)}")

litellm.RateLimitError: OpenAIException - You exceeded your current quota, please check your plan and billing details.

OpenAI'ın ücretsiz deneme bakiyesi dolduğu veya hesaba kart tanımlanıp bakiye yüklenmediği için model isteklere cevap vermeyi durdurdu.

**Model-Agnostic Yapı**:

DSPy sayesinde altyapıdan tamamen izole bir sistem kuruyoruz. OpenAI kotası bittiğinde sadece model ismini ve api anahtarını değiştirerek  Groq/Llama altyapısına geçebildik.

**DSPy MİMARİSİ (Signature)**


Geleneksel yapay zeka kullanımında modele "Şu metni oku, şunları çıkar, şu formatta ver" diye uzun paragraflar yazılır. Burada ise Signature sınıfı ile modele bir protokol dayatıyorsun: "Girdin bu (InputField), çıktın şunlar olmalı (OutputField).

In [36]:
# Signature: Uygulamanın 'Veri Şeması'nı tanımlar.
# Geleneksel prompt mühendisliği yerine, giriş ve çıkışların rollerini belirler.

# 1. Aşama: "Metindeki somut iddialar neler?" sorusuna odaklanmasını sağlıyoruz (ön eleme)
class FactExtractorSignature(dspy.Signature):
    """Haber metnindeki somut iddiaları ve isimleri listeler."""
    news_text = dspy.InputField(desc="Ham haber metni")
    claims = dspy.OutputField(desc="Metindeki doğrulanabilir ana iddiaların listesi")



# 2. Aşama: İlk aşamadan gelen temizlenmiş iddiaları alıp; etik, yasal, clickbait ve teknik özet gibi 5 farklı profesyonel kriterde inceletiyoruz.
class NewsAdvancedSignature(dspy.Signature):
    """
    Haber metnini 360 derece analiz eder.
    KURAL: Analiz yaparken mutlaka 'Rationale' kısmında adım adım mantıksal bir çıkarım yapmalısın.
    """

    # InputField: Modelin işleyeceği ham veri girişini tanımlar.
    news_text = dspy.InputField(desc="Ham haber metni")

    # OutputField: Modelden beklenen yapılandırılmış çıktıları tanımlar.
    summary = dspy.OutputField(desc="Maksimum 100 karakterlik teknik özet")
    legal_risk = dspy.OutputField(desc="Metindeki olası yasal riskler veya etik sorunlar")
    clickbait_score = dspy.OutputField(desc="1-10 arası yanıltıcı başlık puanı")
    fact_check_note = dspy.OutputField(desc="Metindeki şüpheli veya bilim dışı ifadeler")
    confidence = dspy.OutputField(desc="Analize olan güven (0.0 - 1.0)")



**DSPy MİMARİSİ: ÇOK AŞAMALI AKIL YÜRÜTME MODÜLÜ (MULTI-STAGE PIPELINE)**

Signature 'İmzalar' ile protokolleri belirledik. Şimdi bu kuralların hangi sırayla çalışacağını, yani yapay zekanın 'mantık yürütme hattını' kuruyoruz.

In [37]:
# Module: Uygulamanın 'Mantıksal Akışı'nı (Pipeline) tanımlar.
class ProfessionalNewsAgent(dspy.Module):
    def __init__(self):
        super().__init__()
        # İki farklı düşünce zinciri (CoT) modülü tanımlıyoruz
        #dspy.ChainOfThought ile yapay zeka sadece cevap vermez neden böyle düşündüğünü de açıkla
        self.extractor = dspy.ChainOfThought(FactExtractorSignature)
        self.final_auditor = dspy.ChainOfThought(NewsAdvancedSignature)

    def forward(self, news_text):
        # AŞAMA 1: İddia Ayıklama
        extracted = self.extractor(news_text=news_text)
        # AŞAMA 2: Bağlamsal Denetim (Contextual Auditing)
        context = f"İddialar Listesi: {extracted.claims}"
        return self.final_auditor(news_text=f"{context} \n\n Ham Metin: {news_text}")

# Ana bot nesnesini oluştur
bot = ProfessionalNewsAgent()

**DSPy OPTIMIZER (TELEPROMPTER)**

Bu kısım sistemin 'kendi kendine öğrenme' yeteneğidir. Örneklere bakarak pipeline'ı matematiksel olarak optimize eder.

In [38]:
from dspy.teleprompt import BootstrapFewShot

# örnek veri seti
def get_optimized_bot():
    trainset = [
        dspy.Example(news_text="Mars'ta Göktürkçe anıt bulundu.",
                     summary="Asılsız Mars-Göktürk iddiası.",
                     confidence="0.2").with_inputs('news_text'),
        dspy.Example(news_text="Sivas Cumhuriyet Üniversitesi yeni bir AI laboratuvarı açtı.",
                     summary="Sivas'ta akademik gelişim.",
                     confidence="0.9").with_inputs('news_text')
    ]

    # BootstrapFewShot kullanarak sisteme birkaç 'doğru/yanlış' örneği gösterdik
    # DSPy de bu örneklere bakarak en yüksek başarıyı getiren analiz yöntemini kendi kendine derledi (optimize etti)
    optimizer = BootstrapFewShot(metric=None)
    return optimizer.compile(ProfessionalNewsAgent(), trainset=trainset)


bot = get_optimized_bot()

100%|██████████| 2/2 [00:00<00:00,  5.59it/s]

Bootstrapped 2 full traces after 1 examples for up to 1 rounds, amounting to 2 attempts.


In [39]:
# ANALİZ VE AKILLI KARAR MEKANİZMASI

def master_analysis(text):
    #loglama
    current_time = datetime.datetime.now().strftime('%H:%M:%S')
    print(f"\n{'='*70}\n🚀 [ {current_time} ] ANALİZ SÜRECİ BAŞLATILDI\n{'='*70}")

    # Eğer metin boşsa veya çok kısaysa işlemi durduruyoruz
    if not text or len(text.strip()) < 10:
        return "Metin kısa!", "0/10", "Yok", "Yok", "%0", "Girdi yetersiz.", "⚠️ Lütfen geçerli bir metin girin."

    try:
        # Kurduğumuz 2 aşamalı botu burada tetikliyoruz
        res = bot(news_text=text)

        # Veri Ayrıştırma ve Temizleme (Sanitization)
        summary = getattr(res, 'summary', "Özet yok.")
        click_score = str(getattr(res, 'clickbait_score', '0')).split('/')[0].strip()
        legal = getattr(res, 'legal_risk', "Risk yok.")
        fact = getattr(res, 'fact_check_note', "Not yok.")

        # Muhakeme Süreçlerini Birleştirme
        auditor_rationale = getattr(res, 'rationale', getattr(res, 'reasoning', "Analiz yapıldı."))
        full_logic = f"--- 1. AŞAMA (İddia Çıkarımı) ---\nSistem metindeki temel iddiaları ayrıştırdı.\n\n--- 2. AŞAMA (Eleştirel Analiz) ---\n{auditor_rationale}"

        # Güven Skoru Hesaplama ve Karar Verme
        conf_raw = str(getattr(res, 'confidence', '0.5'))
        try:
            clean_val = "".join(c for c in conf_raw if c.isdigit() or c == ".").strip('.')
            conf_f = float(clean_val)
            if conf_f > 1: conf_f /= 100
            conf_display = f"%{conf_f*100:.1f}"
        except:
            conf_f = 0.5
            conf_display = f"%{conf_f*100:.0f}"

        # AKILLI KARAR MOTORU (🔴/🟡/🟢)
        if conf_f < 0.45:
            decision = "🔴 KRİTİK: Model bu habere güvenmiyor! Dezenformasyon riski yüksek."
        elif conf_f < 0.75:
            decision = "🟡 DİKKAT: Metin tutarlı ancak ek doğrulama gerektiriyor."
        else:
            decision = "🟢 GÜVENİLİR: Metin yüksek teknik tutarlılık içeriyor."

        # KONSOL LOGLARI
        print(f"✅ ANALİZ TAMAMLANDI | Güven: {conf_display} | Karar: {decision}")

        return summary, f"Skor: {click_score}/10", legal, fact, conf_display, full_logic, decision

    except Exception as e:
        print(f"❌ SİSTEM HATASI: {str(e)}")
        return "Hata!", "0", "Hata", "Hata", "%0", str(e), "❌ Kritik hata oluştu!"

In [40]:
with gr.Blocks(theme=gr.themes.Soft(), css=".gradio-container {max-width: 95% !important;}") as demo:
    gr.Markdown("""
    # 🛡️ Yapay Zeka Haber Denetçisi (AI News Auditor)
    **Mimari:** `İddia Çıkarımı` ➔ `Haber Analizi` | **Optimizasyon:** `BootstrapFewShot (DSPy)`
    *Bu sistem, haberleri sadece özetlemez; iddiaları ayrıştırır, çapraz sorgular ve yapay zeka güvencesiyle denetler.*
    """)

    with gr.Row():
        txt_input = gr.Textbox(
            label="Analiz Edilecek Haber Metni",
            lines=10,
            placeholder="Şüpheli veya doğrulanması gereken haber metnini buraya yapıştırın..."
        )

    # Akıllı Durum Bildirimi (Karar Mekanizması)
    out_decision = gr.Markdown("### 🕒 Analiz için haber metni girişi bekleniyor...")

    with gr.Row():
        out_sum = gr.Textbox(label="📌 Teknik Özet", lines=5, interactive=False)
        out_click = gr.Textbox(label="⚠️ Tık Tuzağı (Clickbait) Analizi", lines=5, interactive=False)

    with gr.Row():
        out_legal = gr.Textbox(label="⚖️ Etik ve Yasal Risk Değerlendirmesi", lines=6, interactive=False)
        out_fact = gr.Textbox(label="🔍 Doğruluk Kontrolü (Fact-Check) Notu", lines=6, interactive=False)
        out_conf = gr.Textbox(label="📈 Yapay Zeka Güven Skoru", interactive=False)

    # DSPy'ın en güçlü özelliği olan Rationale kısmı
    out_logic = gr.Textbox(
        label="🧠 İki Aşamalı Akıl Yürütme ve Karar Mantığı (Rationale)",
        lines=12,
        interactive=False
    )

    btn = gr.Button("🔍 Derinlemesine Analizi Başlat", variant="primary", size="lg")

    # Fonksiyon bağlama
    btn.click(
        fn=master_analysis,
        inputs=txt_input,
        outputs=[out_sum, out_click, out_legal, out_fact, out_conf, out_logic, out_decision]
    )

demo.launch(share=True, debug=True)

/tmp/ipykernel_21791/3923704240.py:1: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(), css=".gradio-container {max-width: 95% !important;}") as demo:
/tmp/ipykernel_21791/3923704240.py:1: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(), css=".gradio-container {max-width: 95% !important;}") as demo:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://b2dfb80cf74dae47f9.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)



🚀 [ 20:50:42 ] ANALİZ SÜRECİ BAŞLATILDI
✅ ANALİZ TAMAMLANDI | Güven: %10.0 | Karar: 🔴 KRİTİK: Model bu habere güvenmiyor! Dezenformasyon riski yüksek.

🚀 [ 20:53:42 ] ANALİZ SÜRECİ BAŞLATILDI
✅ ANALİZ TAMAMLANDI | Güven: %70.0 | Karar: 🟡 DİKKAT: Metin tutarlı ancak ek doğrulama gerektiriyor.
Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://b2dfb80cf74dae47f9.gradio.live


FLAŞ GELİŞME: NASA'nın gizlediği rapor ortaya çıktı! Mars'ta bulunan bir mağarada üzerinde Göktürkçe yazılar olan bir anıt keşfedildi. Bilim insanları, Türklerin binlerce yıl önce Mars'ta koloni kurduğunu ve radyasyondan korunmak için yer altına çekildiğini iddia ediyor. Bu bilgi dünyadan saklanıyor!


Sivas Cumhuriyet Üniversitesi, yapay zeka alanında uluslararası bir başarıya imza attı. Bilgisayar mühendisliği bölümü öğrencileri, tarımsal verimliliği artıran yeni bir NLP (Doğal Dil İşleme) modeli geliştirdiklerini duyurdu. Proje, tarım sektöründeki veri analiz süreçlerini %30 hızlandırmayı hedefliyor.
